# 001：无微调基线与失败分析（单 Agent）

目标：建立符合比赛接口的单 Agent、无 LoRA 基线，区分环境故障 / 定位失败 / 错误修改 / 工具失败 / 预算耗尽。

改编自官方 Getting Started v2（`ryanholbrook/getting-started-gemma-4-developer-agent`，本次核实 current_version_number=2）。
官方 Notebook 只是开发载体，不复制其文件进本项目；运行时从官方挂载路径重新取得并核对版本。

顺序：环境 -> 数据与基线清单 -> 模型服务 -> 逐题执行 -> 结果与失败分类 -> 打包。先跑 3 个 smoke，再扩大到 `research/task_splits.json` 的 dev_panel（24 题）。

本机（Mac）只做编辑与校验；安装与 GPU 运行只在 Kaggle Linux Notebook 执行，不要把 Linux GPU 安装流程套在 Mac 上。

官方 v2（current_version_number=2，last_run 2026-09-25T07:06:36Z）：相对 v1 仅 3 处实质变更——删 §5 预算说明 cell、VllmConfig 删 `default_chat_template_kwargs`、§5 预算改从 `eval_config.yaml` 全读。本 notebook 同 v2（预算全显式读、不加 thinking 模板参数）。


## 0. 路径与版本记录（先运行）

记录 Python / CUDA / GPU / 关键包实际版本；后续单元格依赖此处变量。

In [ ]:
import hashlib
import json
import platform
import sys
from pathlib import Path

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WHEELHOUSE_SRC = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# 本项目 agent/ 在 Kaggle 中的位置：优先 /kaggle/working/agent（用户上传），其次相对路径 agent/（repo 模式）
CANDIDATES = [Path('/kaggle/working/agent'), Path('agent'), Path('/kaggle/working/agent_001')]
PROJECT_AGENT_SRC = next((p for p in CANDIDATES if (p / 'agent.yaml').exists()), None)
AGENT_DIR = WORKING_DIR / 'agent_run'
print(f'python={sys.version.split()[0]} platform={platform.platform()}')
print(f'DATA_DIR exists={DATA_DIR.exists()} MODEL_PATH exists={MODEL_PATH.exists()}')
print(f'PROJECT_AGENT_SRC={PROJECT_AGENT_SRC}')
try:
    import torch
    print(f'torch={torch.__version__} cuda_available={torch.cuda.is_available()} gpu_count={torch.cuda.device_count() if torch.cuda.is_available() else 0}')
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f'  gpu{i}: {p.name} total_mem={p.total_memory/2**30:.1f}GiB')
except Exception as e:
    print(f'torch info unavailable: {e}')
for pkg in ['swegemma', 'adk_submission', 'adk_eval_core', 'google.adk', 'vllm']:
    try:
        m = __import__(pkg)
        print(f'{pkg}={getattr(m, "__version__", "installed(no __version__)")}')
    except Exception as e:
        print(f'{pkg} NOT imported yet ({e})')


## 1. 环境安装（Kaggle Linux only）

复用官方 v2：去 cutlass `.pth` 钩子、修复 `cu128` wheel 文件名、`--no-deps --force-reinstall` 安装 wheelhouse。以实际挂载路径为准。

In [ ]:
import glob
import importlib
import os
import subprocess
import sys
from pathlib import Path

assert sys.platform.startswith('linux'), '此单元格只在 Kaggle Linux GPU Notebook 运行，Mac 上跳过'
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_SRC.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')


## 2. 数据与基线清单校验

核对 `tasks.jsonl` 指纹、HARNESS 大小、基线 6 工具 / 无 adapter / 无子 Agent、四预算字段显式填写、采样无 `thinking_level`。

In [ ]:
import hashlib
import json
import shutil
import yaml
from pathlib import Path
yaml.SafeLoader.add_constructor('!include', lambda l, n: l.construct_scalar(n))

assert PROJECT_AGENT_SRC is not None, '未找到 agent/：请把本项目 agent/ 上传到 /kaggle/working/agent 后重跑 cell 0'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
shutil.copytree(PROJECT_AGENT_SRC, AGENT_DIR)
print(f'AGENT_DIR={AGENT_DIR}')

# 数据版本（与 docs/plan/README 指纹比对，不一致则记录后采用新版本）
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
raw = TASKS_PATH.read_bytes()
print(f'tasks.jsonl bytes={len(raw)} sha256={hashlib.sha256(raw).hexdigest()}')
print(f'HARNESS_README bytes={(DATA_DIR / "HARNESS_README.md").stat().st_size}')
print(f'snapshots={(DATA_DIR / "snapshots").exists()} graphs={(DATA_DIR / "graphs").exists()} embeddings={(DATA_DIR / "embeddings").exists()} wheels={(DATA_DIR / "wheels").exists()}')

from swegemma.models import load_tasks
tasks = load_tasks(TASKS_PATH)
print(f'Loaded {len(tasks)} tasks')

# 基线结构校验：单 Agent、6 工具、无 adapter、无子 Agent 引用
agent_yaml = (AGENT_DIR / 'agent.yaml').read_text(encoding='utf-8')
cfg = yaml.safe_load(agent_yaml)
assert 'adapter' not in cfg, '001 基线不设置 adapter'
tools = cfg.get('tools', [])
tool_names = [t if isinstance(t, str) else str(t) for t in tools]
assert not any('agent_tool' in t or 'sub_agents' in t for t in tool_names), '001 删除子 Agent 引用'
assert set(tool_names) == {'run_command', 'read_file', 'edit_file', 'write_file', 'get_status', 'submit_patch'}, f'001 只注册6工具，实际={tool_names}'
assert '!include' in agent_yaml and 'prompts/system.md' in agent_yaml, 'instruction 必须 !include prompts/system.md'
print(f'agent tools OK: {tool_names}')

sampling = yaml.safe_load((AGENT_DIR / 'configs' / 'sampling.yaml').read_text(encoding='utf-8'))
assert 'thinking_level' not in str(sampling), '省略 thinking_level'
assert sampling.get('temperature') == 0.2 and sampling.get('top_p') == 0.95, '采样沿用官方 v2 初值'
assert sampling.get('max_output_tokens') == 16384 and sampling.get('thinking_config', {}).get('thinking_budget') == 4096, '输出/思考预算检查'
print('sampling OK')

eval_raw = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text(encoding='utf-8'))
ev = eval_raw.get('evaluation', eval_raw)
for k in ['timeout_seconds', 'max_tool_calls', 'max_time_minutes', 'max_turns']:
    assert k in ev, f'eval_config 缺少 {k}（必须显式填写，不依赖默认）'
print(f'eval budgets: {ev}')

# 任务划分：只读 ID/仓库/commit/标签，不读 patch/test_patch
SPLITS_CANDIDATES = [Path('/kaggle/working/task_splits.json'), Path('research/task_splits.json')]
SPLITS_PATH = next((p for p in SPLITS_CANDIDATES if p.exists()), None)
by_id = {t.instance_id: t for t in tasks}
if SPLITS_PATH is not None:
    splits = json.loads(SPLITS_PATH.read_text(encoding='utf-8'))
    print(f'splits from {SPLITS_PATH}: seed={splits["seed"]} sha={splits["source_sha256"]} counts={splits["counts"]}')
    dev_panel = [by_id[i] for i in splits['dev_panel_ids'] if i in by_id]
    smoke = [by_id[i] for i in splits['smoke_ids'] if i in by_id]
else:
    print('WARNING: 未找到 task_splits.json，先用 tasks[:3] 做环境冒烟，正式对照必须用划分文件')
    dev_panel, smoke = tasks[:24], tasks[:3]
print(f'dev_panel={len(dev_panel)} smoke={[t.instance_id for t in smoke]}')


## 3. 启动模型服务（真实接口）

`validate_single_declared_model` + `discover_adapters`（001 应为 0 个）+ `VllmConfig/VllmServer`，`tool_call_parser=gemma4`，`reasoning_parser=gemma4`，`max_model_len=32768`，tp 按实际 GPU 数选择。模型服务启动时间单独记录。

In [ ]:
import time
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
t0 = time.time()
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
manifest_inner = getattr(adapters, 'adapters', getattr(adapters, 'manifest', None))
if manifest_inner is None:
    manifest_inner = getattr(adapters, '__dict__', None) or {}
n_adapters = len(manifest_inner) if hasattr(manifest_inner, '__len__') else 0
print(f'adapter_manifest_type={type(adapters).__name__} n_adapters={n_adapters}')
assert n_adapters == 0, f'001 无 LoRA，实际发现 {adapters}'
print(f'declared_model={declared_model} adapters={adapters}')
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768, dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto', gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128, tensor_parallel_size=tp_size, startup_timeout=60*20)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM started on {server_instance.base_url} tp={tp_size} startup={time.time()-t0:.1f}s')
models = server_instance.create_model_registry(aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')


## 4. 逐题执行（顺序，Phase1+Phase2）

顺序执行；单题 Agent 计时不含准备，12h 总预算含准备。先跑 smoke（3 题），再跑 dev_panel。`patch`/`test_patch` 不拼入提示。

In [ ]:
import asyncio
import concurrent.futures
import time
import yaml
import pandas as pd
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator

def run_sync(coro_or_fn, *args, **kwargs):
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())

raw_eval = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text(encoding='utf-8'))
sec = raw_eval.get('evaluation', raw_eval)
timeout_seconds = int(sec['timeout_seconds']); max_tool_calls = int(sec['max_tool_calls'])
max_time_minutes = float(sec['max_time_minutes']); max_turns = int(sec['max_turns'])
print(f'budgets: timeout={timeout_seconds}s calls={max_tool_calls} time={max_time_minutes}min turns={max_turns}')

# 默认只跑 smoke；把 RUN_SCOPE 改为 'dev_panel' 再扩大。一次只跑一批，不跨批继承补丁/轨迹。
RUN_SCOPE = 'smoke'
SELECTED = smoke if RUN_SCOPE == 'smoke' else dev_panel
print(f'RUN_SCOPE={RUN_SCOPE} n={len(SELECTED)}')
limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=WORKING_DIR / 'results', submission_dir=AGENT_DIR, models=models, sandbox='subprocess', timeout_seconds=timeout_seconds, max_time_minutes=max_time_minutes, max_tool_calls=max_tool_calls, max_turns=max_turns, limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters, context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10), events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5), graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'), wheels_dir=DATA_DIR / 'wheels', verbose=False)
evaluator = Evaluator(eval_config)
prep_start = time.time()
predictions = []
for idx, task in enumerate(SELECTED, start=1):
    print(f'[{idx}/{len(SELECTED)}] {task.instance_id} ({task.repo})...')
    r = run_sync(evaluator.evaluate_task, task=task, task_index=idx, total_tasks=len(SELECTED))
    predictions.append({'id': task.instance_id, 'prediction': r.agent_patch or ''})
    print(f'  -> resolved={r.resolved} exit={r.test_exit_code} patch_chars={len(r.agent_patch or "")} calls={r.tool_calls} dur={r.duration_seconds:.1f}s')
print(f'prep+run wall={(time.time()-prep_start)/60:.1f}min (不含后续验证拆分以日志为准)')
pd.DataFrame(predictions, columns=['id', 'prediction']).head()


## 5. 结果读取与失败分类（保留原始，再分析）

原始：`summary.json`、`task_results.jsonl`、`patches/<id>.patch`、`test_outputs/<id>.log`、`traces/trace_<id>.json`、`logs/<id>.log`。
实际读取属性：`resolved`、`test_exit_code`、`agent_patch`、`tool_calls`、`duration_seconds`。缺失记空，不推测补值。
失败主因多标签：环境/依赖、定位、理解与修改、工具格式或执行、上下文截断、预算、提交失败、尚无法确定。

In [ ]:
import json
from pathlib import Path
import pandas as pd

RESULTS_DIR = WORKING_DIR / 'results'
rows = []
tr_path = RESULTS_DIR / 'task_results.jsonl'
if tr_path.exists():
    for line in tr_path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            j = json.loads(line)
            rows.append({'instance_id': j.get('instance_id', ''), 'resolved': j.get('resolved', ''), 'test_exit_code': j.get('test_exit_code', ''), 'patch_chars': len(j.get('agent_patch') or ''), 'patch_nonempty': bool((j.get('agent_patch') or '').strip()), 'tool_calls': j.get('tool_calls', ''), 'duration_seconds': j.get('duration_seconds', ''), 'exit_reason': j.get('exit_reason', '') or j.get('status', ''), 'primary_cause': '', 'env_note': ''})
df = pd.DataFrame(rows)
print(f'rows={len(df)} resolved_sum={(df["resolved"]==True).sum() if len(df) else 0}')
display(df.head(30) if len(df) else 'no task_results.jsonl yet')
out_csv = WORKING_DIR / 'analysis_001.csv'
if len(df):
    df.to_csv(out_csv, index=False)
    print(f'saved {out_csv}')
print('按 docs/plan/001 逐题补 primary_cause（人工）：环境/依赖、定位、理解与修改、工具格式或执行、上下文截断、预算、提交失败、尚无法确定')


## 6. 打包（以 `agent/` 为根）

`submission.zip` 根见 `agent.yaml`；`!include` 目标须在包内；不带子 Agent/adapter；校验扩展名与 3GiB。打包成功 != 可执行 != 验证通过 != 正式得分。

In [ ]:
import shutil
import zipfile
from pathlib import Path
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    names = [i.filename for i in infos]
    assert 'agent.yaml' in names, f'包根必须有 agent.yaml，实际={names[:10]}'
    assert not any(n.startswith('sub_agents/') or n.startswith('adapters/') for n in names), '001 不带 sub_agents/adapters'
    total = sum(i.file_size for i in infos)
    assert total <= MAX_SUBMISSION_SIZE_BYTES, f'超过3GiB: {total}'
    for info in infos:
        if not info.is_dir():
            assert Path(info.filename).suffix.lower() in ALLOWED_SUBMISSION_EXTENSIONS, f'非法扩展名: {info.filename}'
print(f'Created {zip_path} ({zip_path.stat().st_size/2**20:.2f} MiB) files={len(names)}')
print(names)


## 7. 运行观察清单（人工回填到 docs/progress/001-baseline.md）

- 模型是否正确加载、工具调用是否完整、补丁能否提取、轨迹与验证输出能否对应同一任务。
- 准备 vs Agent vs 验证耗时分段（仅日志确实提供时填）。
- 环境诊断（import 路径、依赖版本、原始错误）与官方环境说明比对；未修改版本/参考修复的观察属环境诊断，不计 Agent 成绩。
- 报告保留原始分母；不静默剔除困难任务。